# Day 88: An alarm that spots the odd dataset

**Goal:** on Day 87 I found the strange dataset (GH-GhaSP) by hand. Today I teach the computer to find it, so next time it shouts for me.

**The picture:** five classes at school. Four are around 150 cm tall and one is 450 cm. A rule can catch that: *shout if a class is far from the typical class.*

> Restart the kernel, then run top to bottom.

## 0. Setup
`agronomic_functions.py` and `db_config.py` live two folders up, so I add that folder to the search path **before** importing.

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
from agronomic_functions import build_flat, quality_flags, dataset_alarm
from db_config import run_query

## 1. The three moves (practised on made-up data)

1. **One typical number per group:** `groupby(group)[column].median()` gives a short list (a *Series*), one number per dataset.
2. **The typical group:** the **median of those medians**. One odd group drags a mean, but hardly moves a median.
3. **Divide each group by the typical group.** A ratio of 1 means "like the typical group". A ratio of 10 means "ten times bigger".

Then keep only the groups whose ratio is **above** the threshold.

In [ ]:
shops = pd.Series({"north": 10, "east": 12, "west": 11, "south": 100})

typical = shops.median()          # 11.5
ratio = shops / typical
print(ratio.round(1))
print(shops[ratio > 3])           # only 'south'

### Things I got wrong first, and fixed
- **Reading a Series:** the left part is the **name**, the right part is the **value**. Say "name = value" (`S = 210`, not "group T which is 210").
- **Reasons need numbers:** "10 is not above 15", not "it is more".
- **`>` against `>=`:** a ratio of exactly 5 is *not above* 5, so it is not flagged at threshold 5.
- **`NaN`:** `NaN > 3` is `False`, so a group with a missing median is never flagged. Missing is not suspicious.

In [ ]:
d = pd.Series({"A": 5, "B": 6, "C": 5.5, "D": 30, "E": 28})
typical = d.median()                                  # 6
ratio = d / typical
print(ratio.round(2))

for threshold in (4, 4.8, 5):
    print(f"threshold {threshold}:", list(d[ratio > threshold].index))
# 4   -> ['D', 'E']   (5.00 and 4.67 are above 4)
# 4.8 -> ['D']        (4.67 is not above 4.8)
# 5   -> []           (5.00 is not above 5)

## 2. The function
It follows the same style as `quality_flags`: **check the input, do the work, return.**

It **returns** the flagged groups and does not just print them. It is a vending machine that gives me the snack, and not one that only announces "snack ready". A returned value can be saved, counted and used again. A printed one disappears.

This lives in `csa_soil_project/agronomic_functions.py`. It is shown here so the notebook is complete.

**Improvement after review:** the guard also catches a `NaN` typical value (for example, a column that is entirely missing). Before, `NaN == 0` is `False`, so the guard would not have fired and the function would have quietly returned an empty result that looks like "all fine".

In [ ]:
def dataset_alarm(df, column, group="dataset_id", ratio=5):
    # Guards
    if column not in df.columns:
        raise ValueError(f"Column {column} not found in table.")
    if group not in df.columns:
        raise ValueError(f"Group {group} not found in table.")

    # Move 1: median of each group
    medians = df.groupby(group)[column].median()

    # Move 2: the typical group
    typical = medians.median()

    # Guard: dividing by 0 or by a missing number is meaningless
    if pd.isna(typical) or typical == 0:
        raise ValueError("Typical value is 0 or missing. Cannot calculate ratio.")

    # Move 3: divide each group by the typical group
    ratios = medians / typical

    # Return only the flagged groups
    return medians[ratios > ratio]

## 3. The test
Made-up table. Medians: A = 5, B = 6, C = 110, so the typical value is 6 and C is about 18 times typical.

**Lesson about tests:** a test that only *prints* "FAIL" cannot stop anything. The last line would still say `all tests passed`. To make a test able to fail, `raise AssertionError` inside the `try` (it is not a `ValueError`, so `except ValueError` does not swallow it).

**Prove a test can fail:** temporarily remove a check inside the function and watch the test go red. Then put the check back.

In [ ]:
def test_dataset_alarm():
    toy = pd.DataFrame({
        "dataset_id": ["A", "A", "A", "B", "B", "B", "C", "C", "C"],
        "value":      [4, 5, 6,   5, 6, 7,   100, 110, 120]
    })

    # Check 1: only C is flagged
    result = dataset_alarm(toy, "value")
    assert list(result.index) == ["C"], "C should be the only flagged dataset"

    # Check 2: a very high ratio flags nothing
    result_high = dataset_alarm(toy, "value", ratio=50)
    assert len(result_high) == 0, "Nothing should be flagged at ratio 50"

    # Check 3: a missing column raises ValueError
    try:
        dataset_alarm(toy, "wrong_column")
        raise AssertionError("Should have raised ValueError for a missing column")
    except ValueError:
        pass    # good: the alarm rang

    # Check 4: a typical value of 0 raises ValueError (no dividing by zero)
    zero = pd.DataFrame({
        "dataset_id": ["A", "A", "A", "B", "B", "B", "C", "C", "C"],
        "value":      [0, 0, 0,   0, 0, 0,   5, 6, 7]
    })
    try:
        dataset_alarm(zero, "value")
        raise AssertionError("Should have raised ValueError when typical is 0")
    except ValueError:
        pass

    print("all tests passed")

test_dataset_alarm()

## 4. The real table
Rebuild `traced` (flat table with flags and `dataset_id`), exactly as on Day 87.

In [ ]:
flat = build_flat("orgc", ["ph", "clay", "sand"])
checked = quality_flags(flat)
meta = run_query("SELECT layer_id, dataset_id FROM wosis_orgc WHERE country_name = 'Ghana'")
traced = pd.merge(checked, meta, on="layer_id", how="left")
print(len(traced))        # must be 1756

**Predict before running** (write it here first, then run):

| column | will any dataset be flagged? |
|---|---|
| orgc | ? |
| ph | ? |
| clay | ? |
| sand | ? |

In [ ]:
for col in ["orgc", "ph", "clay", "sand"]:
    print(f"--- {col.upper()} ---")
    print(dataset_alarm(traced, col))

**What I found:**
- `orgc`: only **GH-GhaSP = 103.4** is flagged. The alarm finds, on its own, what I found by hand on Day 87.
- `ph`, `clay`, `sand`: nothing flagged.

An alarm must do both: ring when it should, and stay quiet when it should.

## 5. Limits (so I do not misuse the tool)
- **Ratio rules suit values like carbon or clay %**, where "10 times bigger" means something.
- **pH is a log scale**, so the ratio rule is weak for pH. A dataset with median pH 7 against a typical 5.5 has a ratio of only 1.27, yet its hydrogen-ion level is about 30 times lower. For pH, compare **differences** (`medians - typical`) and not ratios.
- With very few datasets, the "typical" group is shaky. Treat the alarm as a pointer to look closer, not as proof of a problem.

## 6. What I keep from today
- Read a Series as **name = value**.
- Back every reason with numbers.
- A guard clause stops silent wrong answers (zero or missing "typical" value).
- A test must be able to **fail**; prove it once.
- **Return** results. Do not only print them.
- No tool is universal: know the assumption behind it.

**Next:** Day 89, a small cleaning report that gathers every finding (144 flags, GH-GhaSP carbon, missing clay and sand) into one table someone else could read.